# Cross rank stability analysis

Reads `allen_classified.csv` from `main_results/allen_posterior_mass_analysis/` 
(produced by `analysis/posterior_mass_analysis/02_classify_particles.ipynb`).

**Metric:** `neg_nle = −log_weight = neg_log_marginal_NLE` — lower = better fit.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from modelsmc.utils.plot_utils import use_style

FIGURES_DIR = Path("../fig")
FIGURES_DIR.mkdir(exist_ok=True)

CLASSIFIED_CSV = Path(
    "../../../main_results/allen_posterior_mass_analysis/allen_classified.csv"
)
assert CLASSIFIED_CSV.exists(), "Run 02_classify_particles.ipynb first."
print(f"Figures → {FIGURES_DIR.resolve()}")

In [ ]:
df = pd.read_csv(CLASSIFIED_CSV)
print(f"Loaded {len(df)} classified particles")
print(f"Seeds: {sorted(df['seed'].unique())}")

df["channel_type"] = df["subtype_id"].fillna("unknown").str.strip()

# neg_nle = neg_log_marginal_NLE = -log_weight  (positive; lower = better fit)
df["neg_nle"] = -df["log_weight"]
print(
    f"log_weight range : [{df['log_weight'].min():.1f}, {df['log_weight'].max():.1f}]"
)
print(f"neg_nle   range  : [{df['neg_nle'].min():.1f}, {df['neg_nle'].max():.1f}]")

# ── Outlier filter: upper Tukey fence on neg_nle ──────────────────────────────
q1, q3 = df["neg_nle"].quantile([0.25, 0.75])
upper_fence = q3 + 3.0 * (q3 - q1)
n_out = (df["neg_nle"] > upper_fence).sum()
print(f"Outlier fence (Q3 + 3·IQR): {upper_fence:.1f}  →  {n_out} particles removed")

df_exp = df[df["neg_nle"] <= upper_fence].copy().reset_index(drop=True)
df_exp = df_exp[df_exp["channel_type"] != "unknown"].copy().reset_index(drop=True)
print(f"df_exp: {len(df_exp)} particles after outlier removal and unknown filter")
print("\nChannel type counts:")
print(df_exp["channel_type"].value_counts().to_string())

---
## Analysis — Cross-seed consistency

In [ ]:
# ── Family ranking consistency across seeds ─────────────────────────────────
# family_id is already a column in df_exp (from the LLM classification)
# Uses min neg_nle per (seed, family)

# Per-seed MINIMUM neg_nle per family
family_per_seed = df_exp.groupby(["seed", "family_id"])["neg_nle"].min().reset_index()
family_per_seed = family_per_seed[family_per_seed["family_id"] != "unknown"].copy()

# Pivot: rows = seeds, columns = families
pivot_fam = family_per_seed.pivot(index="seed", columns="family_id", values="neg_nle")

# Rank families within each seed (rank 1 = best = lowest neg_nle)
# NaN (family absent in a seed) gets the worst rank
rank_per_seed = pivot_fam.rank(axis=1, method="min", na_option="bottom")

n_seeds = len(pivot_fam)
# Sort families by their global minimum across seeds (best first)
families = pivot_fam.min().sort_values().index.tolist()
n_fam = len(families)

print(
    f"Family ranking per seed (rank 1 = best = lowest min neg_nle, n = {n_seeds} seeds)"
)
print()
header = f"  {'Family':<10}" + "".join(f"  rank {r}" for r in range(1, n_fam + 1))
print(header)
print("  " + "-" * (len(header) - 2))
for fam in families:
    row = f"  {fam:<10}"
    for rank in range(1, n_fam + 1):
        count = int((rank_per_seed[fam] == rank).sum())
        row += f"  {count:>4}/{n_seeds}"
    print(row)

print()
for fam in families:
    n_best = int((rank_per_seed[fam] == 1).sum())
    print(f"  {fam} ranked 1st in {n_best}/{n_seeds} seeds")

In [ ]:
# ── Rank heatmap figure ─────────────────────────────────────────────────────
# Rows = seeds, columns = families (sorted best→worst by global min).
# Cell colour = rank within that seed (1 = best = darkest).

FAMILY_LABELS = {
    "base_only": "Base",
    "I_M": r"$I_M$",
    "I_M_plus_I_NaP": r"$I_M$+$I_{NaP}$",
    "I_M_plus_I_h": r"$I_M$+$I_h$",
    "I_M_plus_I_A": r"$I_M$+$I_A$",
    "I_M_plus_I_Ks": r"$I_M$+$I_{Ks}$",
    "I_M_plus_I_bgK": r"$I_M$+$I_{bgK}$",
}

# pivot_fam and rank_per_seed are already defined in the cell above
rank_matrix = rank_per_seed[families]  # re-order columns to match `families` sort

_scale = 90 / 72
_fig_w = min(len(families) * 1.2, 8) / 2.54 * _scale  # ~1.2 cm per family column
_fig_h = max(len(rank_matrix) * 0.55, 4) / 2.54 * _scale  # ~0.55 cm per seed row

with use_style("pyloric"):
    fig, ax = plt.subplots(figsize=(_fig_w, _fig_h))

    im = ax.imshow(
        rank_matrix.values,
        cmap="YlOrRd",  # low rank (= best) → light yellow (brighter = better)
        vmin=1,
        vmax=n_fam,
        aspect="auto",
    )

    # Annotate each cell with its rank number
    for row_i, seed in enumerate(rank_matrix.index):
        for col_j, fam in enumerate(families):
            rank_val = int(rank_matrix.loc[seed, fam])
            ax.text(
                col_j,
                row_i,
                str(rank_val),
                ha="center",
                va="center",
                fontsize=8,
                color="black",
            )

    ax.set_xticks(range(len(families)))
    ax.set_xticklabels(
        [FAMILY_LABELS.get(f, f) for f in families], rotation=35, ha="right"
    )
    ax.set_yticks(range(len(rank_matrix)))
    ax.set_yticklabels([f"seed {s}" for s in rank_matrix.index], fontsize=8)
    ax.set_title("Family rank per seed  (1 = best)", pad=6)

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Rank", fontsize=8)
    cbar.set_ticks(range(1, n_fam + 1))

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "family_rank_heatmap.svg", bbox_inches="tight")
    plt.savefig(FIGURES_DIR / "family_rank_heatmap.pdf", bbox_inches="tight")
    plt.show()
print("Saved family_rank_heatmap.svg")

**$I_M$-type channel variants consistently achieve the best weights across independent seeds.**
Rank heatmap for the Allen HH task across 10 independent ModelSMC runs. Each row is a seed and each column is an ion-channel family; cell colour and number give the rank of that family within that seed (1\,=\,best\,=\,lowest minimum NLE weight, brighter\,=\,better). $I_M$ variants rank first in the majority of seeds, confirming that the preference for M-type channels is a stable posterior conclusion of ModelSMC and not an artefact of any single run.